<a href="https://colab.research.google.com/github/amilafr/algo-python-pro2/blob/main/Module_Web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# db_scripts.py

In [ ]:
import sqlite3
db_name = 'quiz.sqlite'
conn = None
cursor = None

def open():
    global conn, cursor
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

def close():
    cursor.close()
    conn.close()

def do(query):
    cursor.execute(query)
    conn.commit()

def clear_db():
    ''' delete all tables '''
    open()
    query = '''DROP TABLE IF EXISTS quiz_content'''
    do(query)
    query = '''DROP TABLE IF EXISTS question'''
    do(query)
    query = '''DROP TABLE IF EXISTS quiz'''
    do(query)
    close()

# bikin tabel
def create():
    open()
    # mengatifkan koneksi
    cursor.execute('''PRAGMA foreign_keys=on''')

    # tabel quiz
    do('''CREATE TABLE IF NOT EXISTS quiz(
            id INTEGER PRIMARY KEY,
            name VARCHAR)
        ''')

    # tabel question
    do('''CREATE TABLE IF NOT EXISTS question(
            id INTEGER PRIMARY KEY,
            question VARCHAR,
            answer VARCHAR,
            wrong1 VARCHAR,
            wrong2 VARCHAR,
            wrong3 VARCHAR
    )''')

    # tabel quiz content
    do('''CREATE TABLE IF NOT EXISTS quiz_content(
            id INTEGER PRIMARY KEY,
            quiz_id INTEGER,
            question_id INTEGER,
            FOREIGN KEY (quiz_id) REFERENCES quiz (id),
            FOREIGN KEY (question_id) REFERENCES question (id)
    )''')
    close()

# nambah data ke tabel question
def add_question():
    questions = [
        ('How many months in a year have 28 days?', 'All', 'One', 'None','Two'),
        ('What will the green cliff look like if it falls into the Red Sea?', 'Wet', 'Red', 'Will not change', 'Purple'),
        ('Which hand is better to stir tea with?', 'With a spoon', 'Right', 'Left', 'Any'),
        ('What has no length, depth, width, or height, but can be measured?', 'Time', 'Stupidity', 'The sea','Air'),
        ('When is it possible to draw out water with a net?', 'When the water is frozen', 'When there are no fish', 'When the goldfish swim away', 'When the ne breaks'),
        ('What is bigger than an elephant and weighs nothing?', 'Shadow of elephant','A balloon','A parachute', 'A cloud')
    ]

    open()
    cursor.executemany('''INSERT INTO question (question, answer, wrong1, wrong2, wrong3) VALUES (?,?,?,?,?)''', questions)
    conn.commit()
    close()

# nambah data ke tabel quiz
def add_quiz():
    quizes = [
        ('Own game', ),
        ('Who wants to be a millionaire?', ),
        ('The smartest', )
    ]

    open()
    cursor.executemany('''INSERT INTO quiz (name) VALUES (?)''', quizes)
    conn.commit()
    close()

# nambah data ke quiz_content
def add_links():
    open()
    cursor.execute('''PRAGMA foreign_keys=on''') # aktifin foreign key
    query = "INSERT INTO quiz_content (quiz_id, question_id) VALUES (?,?)"
    answer = input('Add link? (y/n) : ')
    while answer != 'n':
        quiz_id = int(input('Masukkan quiz id: '))
        question_id = int(input('Masukkan question id : '))
        cursor.execute(query, [quiz_id, question_id])
        conn.commit()
        answer = input('Add link? (y/n) : ')
    close()

def get_question_after(question_id, quiz_id):
    open()
    query = '''SELECT quiz_content.id, question.question, question.answer, question.wrong1, question.wrong2, question.wrong3
                FROM quiz_content, question
                WHERE quiz_content.question_id == question.id
                AND quiz_content.id > ? and quiz_content.quiz_id == ?
                ORDER BY quiz_content.id
    '''
    cursor.execute(query, [question_id, quiz_id])
    result = cursor.fetchone()
    close()

    return result

def show(table):
    query = 'SELECT * FROM ' + table
    open()
    cursor.execute(query)
    print(cursor.fetchall())
    close()

def show_tables():
    show('question')
    show('quiz')
    show('quiz_content')

# tambahan
def get_quises():
    query = 'SELECT * FROM quiz ORDER BY id'
    open()
    cursor.execute(query)
    result = cursor.fetchall()
    close()
    return result

def main():
    clear_db()
    create()
    # tambahkan data
    add_question() # question
    add_quiz() # quiz
    add_links() # quiz_content
    show_tables()

if __name__ == "__main__":
    main()

# quiz.py

In [ ]:
from flask import *
from random import randint
from db_scripts import *

# tambahan
def start_quiz(quiz_id):
    session['quiz'] = quiz_id
    session['last_question'] = 0

def end_quiz():
    session.clear()

# menampilkan quiz dari database
def quiz_form():
    html_beg = '''<html> <body> <h2>Choose a quiz</h2> <form method="post" action="index"> <select name="quiz">'''
    frm_submit = '''<p> <input type="submit" value="Select"> </p>'''
    html_end = '''</select>''' + frm_submit + '''</form> </body> </html>'''

    options = ''' '''

    # list quiz di database --> ditampilin pake dropdown list
    q_list = get_quises()
    for id, name in q_list:
        option_line = ('''<option value="''' + str(id) + '''">''' + str(name) + '''</option>''')
        options = options + option_line

    return html_beg + options + html_end

# modifikasi
def index():
    if request.method == 'GET':
        start_quiz(-1)
        return quiz_form() # nampilin quiz form
    else:
        quiz_id = request.form.get('quiz')
        start_quiz(quiz_id)
        return redirect(url_for('test'))

# modifikasi
def test():
    if not ('quiz' in session) or int(session['quiz']) < 0:
        return redirect(url_for('index'))
    else:
        # global last_question
        result = get_question_after(session['last_question'], session['quiz'])
        if result is None or len(result) == 0:
            return redirect(url_for('result'))
        else:
            session['last_question'] = result[0]
            return '<h1>' + str(session['quiz']) + '<br>' + str(result) + '</h1>'

def result():
    end_quiz()
    return 'Finished'

# bikin app web
app = Flask(__name__)

# daftar url
app.add_url_rule('/', 'index', index) # utama
app.add_url_rule('/index', 'index', index, methods=['post', 'get']) # pake method
app.add_url_rule('/test', 'test', test)
app.add_url_rule('/result', 'result', result)

# kunci enkripsi
app.config['SECRET_KEY'] = 'inikoderahasiabanget'

if __name__ == '__main__':
    app.run() # run app